# 03 — Sentiment Engine
**Goal:** Score each risk-flagged sentence using FinBERT — a transformer model 
pre-trained on financial text — to produce structured sentiment output.

**Input:** `outputs/jpm_nlp_pipeline_2025Q2.json` — 548 risk-flagged sentences  
**Output:** `outputs/jpm_sentiment_2025Q2.json` — sentences with sentiment scores

**Model:** ProsusAI/finbert (HuggingFace) — trained on financial phraseology,  
outperforms general-purpose BERT on earnings call and analyst report language.

In [1]:
import json
import pathlib

input_path = pathlib.Path("../outputs/jpm_nlp_pipeline_2025Q2.json")

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

flagged_sentences = data["flagged_sentences"]
sentences_text = [item["sentence"] for item in flagged_sentences]

print(f"Company: {data['company']}")
print(f"Period:  {data['period']}")
print(f"Sentences to score: {len(sentences_text):,}")
print(f"\nSample:")
print(f"  {sentences_text[0][:200]}")

Company: JPMorgan Chase & Co.
Period:  2025-06-30
Sentences to score: 548

Sample:
  • Total net revenue was $44.9 billion, down 11%, reflecting: – Net interest income ("NII") was $23.2 billion, up 2%, driven by higher Markets net interest income, higher wholesale deposit balances, hi


In [2]:
from transformers import pipeline
import torch

#Check if GPU is available and set device accordingly
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

# Load Finbert sentiment pipeline
finbert = pipeline(
    task = "text-classification",
    model = "ProsusAI/finBERT",
    device = device
)

print("Finbert loaded successfully.")
print(f"Labels: {finbert.model.config.id2label}")

c:\Users\Ozoha\nlp_finance_sentiment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: CPU


c:\Users\Ozoha\nlp_finance_sentiment\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ozoha\.cache\huggingface\hub\models--ProsusAI--finBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6005.69it/s]


Finbert loaded successfully.
Labels: {0: 'positive', 1: 'negative', 2: 'neutral'}


In [3]:
# Sanity check on a known sentence before running 
test_sentences = [
     "The provision for credit losses was $2.8 billion.",
    "Net income was $14.9 billion, up 15% from the prior year.",
    "The Firm remains well-capitalized with a CET1 ratio of 15.4%."
]

for s in test_sentences:
    result = finbert(s, truncation = True, max_length = 512)[0]
    print(f" [{result['label'].upper():<8} {result['score']:.3f}] {s}")

 [NEGATIVE 0.504] The provision for credit losses was $2.8 billion.
 [POSITIVE 0.955] Net income was $14.9 billion, up 15% from the prior year.
 [POSITIVE 0.816] The Firm remains well-capitalized with a CET1 ratio of 15.4%.


In [4]:
from tqdm.auto import tqdm

results = []

BATCH_SIZE = 16

for i in tqdm(range(0, len(sentences_text), BATCH_SIZE), desc = "Scoring"):
    batch = sentences_text[i:i+BATCH_SIZE]
    # Truncate to 512 tokens - finBert's max context window
    scored = finbert(batch, truncation = True, max_length = 512, batch_size = BATCH_SIZE)
    results.extend(scored)

print(f"\nScored: {len(results):,} sentences")
print(f"Sample: {results[0]}")

Scoring: 100%|██████████| 35/35 [01:41<00:00,  2.89s/it]


Scored: 548 sentences
Sample: {'label': 'negative', 'score': 0.9573468565940857}


In [5]:
import pandas as pd

# Attach scores back to original sentence data
scored_sentences = []
for item, result in zip(flagged_sentences, results):
    scored_sentences.append({
        "sentence": item["sentence"],
        "categories": item["categories"],
        "phrases": item["phrases"],
        "sentiment": result["label"],
        "confidence": round(result["score"], 4)
    })

df = pd.DataFrame(scored_sentences)

# Distribution
print("Sentiment distribution across 548 risk-flagged sentences:\n")
dist = df["sentiment"].value_counts()
for label, count in dist.items():
    pct = count / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"  {label.upper():<10} {count:>4}  ({pct:.1f}%)  {bar}")

print(f"\nMean confidence: {df['confidence'].mean():.3f}")
print(f"Low confidence sentences (<0.6): {(df['confidence'] < 0.6).sum()}")

Sentiment distribution across 548 risk-flagged sentences:

  NEUTRAL     437  (79.7%)  ███████████████████████████████████████
  POSITIVE     56  (10.2%)  █████
  NEGATIVE     55  (10.0%)  █████

Mean confidence: 0.871
Low confidence sentences (<0.6): 27


In [6]:
# Look at the most confident positive and negative sentences
print("TOP 5 NEGATIVE sentences (highest confidence):\n")
top_neg = df[df["sentiment"] == "negative"].nlargest(5, "confidence")
for _, row in top_neg.iterrows():
    print(f"  [{row['confidence']:.3f}] {row['sentence'][:200]}")
    print(f"  Categories: {row['categories']}\n")

print("TOP 5 POSITIVE sentences (highest confidence):\n")
top_pos = df[df["sentiment"] == "positive"].nlargest(5, "confidence")
for _, row in top_pos.iterrows():
    print(f"  [{row['confidence']:.3f}] {row['sentence'][:200]}")
    print(f"  Categories: {row['categories']}\n")

TOP 5 NEGATIVE sentences (highest confidence):

  [0.976] • Lending revenue was $1.8 billion, down 6%, largely driven by higher fair value losses on credit protection purchased against certain retained loans and lending-related commitments.
  Categories: ['market_risk']

  [0.976] Net interest income was $3.1 billion, down 35%, driven by the impact of changes in the FTP for consumer deposits and of lower rates, partially offset by the impact of investment securities activity in
  Categories: ['market_risk', 'liquidity_risk']

  [0.976] Net interest income was $1.5 billion, down 37%, driven by the impact of changes in the FTP for consumer deposits and of lower rates, partially offset by the impact of investment securities activity in
  Categories: ['market_risk', 'liquidity_risk']

  [0.971] • $ 1 million of net losses on liabilities, driven by losses in deposits and short-term borrowings predominantly offset by gains in trading liabilities - debt and equity instruments and long-term de

In [7]:
# Compute weighted sentiment score — the headline output of the tool
# Scale: -1.0 (fully negative) to +1.0 (fully positive)

def sentiment_score(row):
    if row["sentiment"] == "positive":
        return row["confidence"]
    elif row["sentiment"] == "negative":
        return -row["confidence"]
    else:
        return 0.0

df["score"] = df.apply(sentiment_score, axis=1)

# Overall filing score
overall_score = df["score"].mean()
positive_score = df[df["sentiment"] == "positive"]["confidence"].mean()
negative_score = df[df["sentiment"] == "negative"]["confidence"].mean()

print("=" * 50)
print(f"  JPMorgan Chase — Q2 2025 MD&A")
print(f"  Overall sentiment score: {overall_score:+.4f}")
print(f"  Positive mean confidence: {positive_score:.4f}")
print(f"  Negative mean confidence: {negative_score:.4f}")
print("=" * 50)

# Score by risk category
print("\nSentiment score by risk category:\n")
for category in ["credit_risk", "market_risk", "macro_signals", "liquidity_risk"]:
    cat_df = df[df["categories"].apply(lambda x: category in x)]
    cat_score = cat_df["score"].mean()
    cat_count = len(cat_df)
    bar = "▓" * int(abs(cat_score) * 20)
    direction = "+" if cat_score > 0 else ""
    print(f"  {category:<20} {direction}{cat_score:.4f}  n={cat_count:<4} {bar}")

  JPMorgan Chase — Q2 2025 MD&A
  Overall sentiment score: +0.0077
  Positive mean confidence: 0.8445
  Negative mean confidence: 0.7832

Sentiment score by risk category:

  credit_risk          -0.0550  n=131  ▓
  market_risk          +0.0353  n=282  
  macro_signals        -0.0698  n=61   ▓
  liquidity_risk       +0.0620  n=167  ▓


In [8]:
# Save final structured output
from datetime import datetime

final_output = {
    "company": data["company"],
    "ticker": data["ticker"],
    "period": data["period"],
    "scored_at": datetime.now().isoformat(),
    "model": "ProsusAI/finbert",
    "summary": {
        "total_scored": len(df),
        "overall_sentiment_score": round(overall_score, 4),
        "distribution": {
            "positive": int((df["sentiment"] == "positive").sum()),
            "negative": int((df["sentiment"] == "negative").sum()),
            "neutral": int((df["sentiment"] == "neutral").sum()),
        },
        "mean_confidence": round(df["confidence"].mean(), 4),
        "by_category": {
            cat: round(
                df[df["categories"].apply(lambda x: cat in x)]["score"].mean(), 4
            )
            for cat in ["credit_risk", "market_risk", "macro_signals", "liquidity_risk"]
        }
    },
    "scored_sentences": df.to_dict(orient="records")
}

out_path = pathlib.Path("../outputs/jpm_sentiment_2025Q2.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final_output, f, ensure_ascii=False, indent=2)

print(f"Saved: {out_path.name}")
print(f"Sentences scored: {len(df):,}")
print("\nStage 3 complete.")

Saved: jpm_sentiment_2025Q2.json
Sentences scored: 548

Stage 3 complete.
